# Lesson 3.1 — The Analytic Jacobian
**Module 6 · Unit 3 · Lesson 9**

The analytic Jacobian differentiates the pose $\mathbf x=[\mathbf p;\boldsymbol\phi]$ (position + roll-pitch-yaw). We confirm its **position rows equal the geometric linear rows**, while its orientation rows are angle-rates (reconciled in L3.2). Convention: ZYX roll-pitch-yaw (pending M2).

In [1]:
import numpy as np
def skew(v):
    v=np.asarray(v,float).ravel(); return np.array([[0,-v[2],v[1]],[v[2],0,-v[0]],[-v[1],v[0],0]])
def vee(S): return np.array([S[2,1],S[0,2],S[1,0]])
def dh(th,d,a,al):
    ct,st,ca,sa=np.cos(th),np.sin(th),np.cos(al),np.sin(al)
    return np.array([[ct,-st*ca,st*sa,a*ct],[st,ct*ca,-ct*sa,a*st],[0,sa,ca,d],[0,0,0,1]])
def forward_chain(P,T,q):
    M=np.eye(4); Ms=[M.copy()]
    for i,(th0,d0,a,al) in enumerate(P):
        th,d=(th0+q[i],d0) if T[i]=="R" else (th0,d0+q[i])
        M=M@dh(th,d,a,al); Ms.append(M.copy())
    return Ms
def geometric_jacobian(P,T,q):
    Ms=forward_chain(P,T,q); on=Ms[-1][:3,3]; J=np.zeros((6,len(q)))
    for i in range(len(q)):
        z=Ms[i][:3,2]; o=Ms[i][:3,3]
        if T[i]=="R": J[:3,i]=np.cross(z,on-o); J[3:,i]=z
        else: J[:3,i]=z
    return J
def pose(P,T,q):
    M=forward_chain(P,T,q)[-1]; return M[:3,3], M[:3,:3]

# ZYX roll-pitch-yaw
def rpy_from_R(R):
    return np.array([np.arctan2(R[2,1],R[2,2]),
                     np.arctan2(-R[2,0],np.hypot(R[2,1],R[2,2])),
                     np.arctan2(R[1,0],R[0,0])])   # [roll, pitch, yaw]
def R_from_rpy(rpy):
    r,p,y=rpy
    Rz=np.array([[np.cos(y),-np.sin(y),0],[np.sin(y),np.cos(y),0],[0,0,1.0]])
    Ry=np.array([[np.cos(p),0,np.sin(p)],[0,1,0],[-np.sin(p),0,np.cos(p)]])
    Rx=np.array([[1,0,0],[0,np.cos(r),-np.sin(r)],[0,np.sin(r),np.cos(r)]])
    return Rz@Ry@Rx

# spatial 3R test arm
ARM=([(0,0,0,np.pi/2),(0,0,1.0,0),(0,0,0.8,0)],["R","R","R"])


## Analytic Jacobian by finite-differencing $[\mathbf p;\,\mathrm{rpy}(R)]$

In [2]:
checks=[]
P,T=ARM; q=np.array([0.4,0.6,-0.3])
def analytic_jacobian(P,T,q,eps=1e-6):
    n=len(q); J=np.zeros((6,n))
    for i in range(n):
        e=np.zeros(n); e[i]=eps
        pp,Rp=pose(P,T,q+e); pm,Rm=pose(P,T,q-e)
        xp=np.hstack([pp,rpy_from_R(Rp)]); xm=np.hstack([pm,rpy_from_R(Rm)])
        J[:,i]=(xp-xm)/(2*eps)
    return J
Ja=analytic_jacobian(P,T,q); Jg=geometric_jacobian(P,T,q)
print("analytic J (rows: 3 position, 3 rpy-rate):\n",np.round(Ja,3))

analytic J (rows: 3 position, 3 rpy-rate):
 [[-0.619 -0.738 -0.218]
 [ 1.464 -0.312 -0.092]
 [ 0.     1.59   0.764]
 [ 0.     0.     0.   ]
 [ 0.    -1.    -1.   ]
 [ 1.     0.     0.   ]]


## Position rows are identical to the geometric linear rows

In [3]:
print("max |J_p - J_v| =",np.max(np.abs(Ja[:3,:]-Jg[:3,:])))
checks.append(np.allclose(Ja[:3,:],Jg[:3,:],atol=1e-5))

max |J_p - J_v| = 1.8807100321538428e-10


## Orientation rows differ (angle-rates, not angular velocity)

In [4]:
diff=np.max(np.abs(Ja[3:,:]-Jg[3:,:]))
print("orientation rows differ; max |J_phi - J_omega| =",round(diff,3))
checks.append(diff>1e-3)   # they are genuinely different objects (reconciled by B in L3.2)

orientation rows differ; max |J_phi - J_omega| = 0.389


In [5]:
assert all(checks), f"FAILED: {checks}"
print("All checks passed.")

All checks passed.
